# L8 · LangGraph con mensajes: `MessagesState`

*Advanced Topics in Analytics II - Generative AI* · Instructor: Sergio A. Mora Pardo

En [L7](L7-LangGraphSimple.ipynb) el estado tenía campos sueltos. Una conversación, en cambio, es una **lista de mensajes** que crece: el usuario escribe, el modelo responde, el usuario vuelve a escribir. LangGraph trae un estado listo para eso, `MessagesState`, con un reducer que **agrega** los mensajes nuevos en lugar de reemplazar la lista.

## Objetivos

1. Repasar las clases de mensajes (`HumanMessage`, `AIMessage`, `SystemMessage`).
2. Entender el reducer `add_messages`: agrega, y reemplaza por `id`.
3. Construir un grafo de un nodo que llama a Claude con el historial.
4. Continuar la conversación pasando el estado anterior.

El recorrido realiza **2 llamadas a Claude Haiku**, en la sección 3.

## 0. Preparación del modelo

In [ ]:
from dotenv import find_dotenv, load_dotenv
from langchain_anthropic import ChatAnthropic

# find_dotenv sube por las carpetas hasta encontrar el .env de la raíz
load_dotenv(find_dotenv())

model = ChatAnthropic(
    model="claude-haiku-5-5",
    max_tokens=500,
    thinking={"type": "disabled"},
    timeout=60,
    max_retries=2,
)

## 1. Los mensajes

Como en L1: `HumanMessage` es lo que escribe la persona, `AIMessage` lo que responde el modelo y `SystemMessage` las instrucciones. Una conversación es una lista de ellos, en orden.

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

history = [
    HumanMessage("Hola, soy el cliente C-1042."),
    AIMessage("Hola, ¿en qué te puedo ayudar?"),
]
for message in history:
    message.pretty_print()

## 2. El reducer `add_messages`

Sin modelo. `MessagesState` es un `TypedDict` con un solo campo, `messages`, y el reducer `add_messages`:

- Si un nodo devuelve mensajes **nuevos**, se **agregan** al final.
- Si devuelve un mensaje con el **mismo `id`** de uno existente, lo **reemplaza**: así se puede corregir un mensaje sin duplicarlo.

Lo probamos directamente, sin grafo:

In [ ]:
from langgraph.graph.message import add_messages

first = HumanMessage("¿Cuánto debo?", id="m1")
current = add_messages([first], [AIMessage("Déjame revisar.", id="m2")])
print([message.text for message in current])

corrected = add_messages(current, [HumanMessage("¿Cuánto debo en mi Crédito Libre Inversión?", id="m1")])
print([message.text for message in corrected])

## 3. Un grafo con el modelo

Un solo nodo, `assistant`, recibe todos los mensajes, les antepone el *system prompt* y llama a Claude. Devuelve **solo el mensaje nuevo**: el reducer lo agrega al historial.

```mermaid
flowchart LR
    S([START]) --> A["assistant<br/>system + messages → Claude"] --> E([END])
```

In [ ]:
from langgraph.graph import END, START, MessagesState, StateGraph

SYSTEM_PROMPT = SystemMessage(
    "Eres el asistente de soporte de ACME, una fintech. Responde en español, en máximo dos frases. "
    "No tienes acceso a la cartera: si te piden un saldo, explica que necesitas consultarlo."
)


def assistant(state: MessagesState) -> dict:
    return {"messages": [model.invoke([SYSTEM_PROMPT, *state["messages"]])]}


builder = StateGraph(MessagesState)
builder.add_node("assistant", assistant)
builder.add_edge(START, "assistant")
builder.add_edge("assistant", END)
chat = builder.compile()

state = chat.invoke({"messages": [HumanMessage("Hola, soy el cliente C-1042. ¿Cuánto debo?")]})
for message in state["messages"]:
    message.pretty_print()

Para seguir la conversación, pasamos el estado anterior con un mensaje nuevo. El grafo **no recuerda nada por sí solo**: el historial viaja en el estado. En L9 vimos que un *checkpointer* lo guarda por `thread_id`.

In [ ]:
state = chat.invoke({"messages": [*state["messages"], HumanMessage("¿Y qué necesitas para consultarlo?")]})
print(len(state["messages"]), "mensajes")
state["messages"][-1].pretty_print()

## Práctica breve

1. Cambia el *system prompt* para que el asistente siempre pida el código del cliente si no lo tiene. Empieza una conversación sin código.
2. Agrega al estado un campo `customer: str` y un nodo previo que lo extraiga del primer mensaje con una expresión regular.
3. Con `add_messages`, reemplaza la respuesta del asistente por otra con el mismo `id`. ¿Cuántos mensajes quedan?

## Referencias

- [`MessagesState` y `add_messages`](https://docs.langchain.com/oss/python/langgraph/graph-api#messagesstate)
- **Learning LangChain**, capítulo 5: [`a-chatbot.py`](https://github.com/langchain-ai/learning-langchain/blob/master/ch5/py/a-chatbot.py)